# **Minimax-H3 for AI Video Generation (ComfyUI)**
- Run the cell below to get a link (e.g. https://literature-consortium-align.trycloudflare.com ) which you can use to launch the comfyUI interface.
- If you get an error on the resize image node, change the upscale_method.
* You can get models and workflows here:
  * (1) https://huggingface.co/Comfy-Org/MiniMax-H3/tree/main
  * (2) https://huggingface.co/realrebelai/MiniMax-H3_GGUFs/tree/main
  * (3) https://huggingface.co/Abiray/MiniMax-H3-GGUF/tree/main  
  * (4) https://comfy.org/workflows/comfyui/
- Huggingface page: https://huggingface.co/MiniMaxAI/MiniMax-H3

- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks

In [ ]:
# @title # 1. 💥 Prepare Environment & Install Dependencies {"single-column":true}
import os

# 1. ALWAYS force the directory back to the Colab root before cloning
%cd /content

# 2. Clone ComfyUI only if it doesn't already exist in the root
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

# 3. Move into the correct, top-level ComfyUI folder
%cd /content/ComfyUI

# Install core dependencies
!pip install -r requirements.txt
from IPython.display import clear_output

# 4. Install custom nodes only if they don't already exist
if not os.path.exists("custom_nodes/ComfyUI-GGUF"):
    !cd custom_nodes && git clone https://github.com/city96/ComfyUI-GGUF.git

# 5. Install Video Helper Suite
if not os.path.exists("custom_nodes/ComfyUI-VideoHelperSuite"):
    !cd custom_nodes && git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    !cd custom_nodes/ComfyUI-VideoHelperSuite && pip install -r requirements.txt

# 6. Install KJNodes
if not os.path.exists("custom_nodes/ComfyUI-KJNodes"):
    !cd custom_nodes && git clone https://github.com/kijai/ComfyUI-KJNodes.git
    !cd custom_nodes/ComfyUI-KJNodes && pip install -r requirements.txt

# Install GGUF and high-speed Hugging Face download engine
!pip install gguf huggingface_hub hf_transfer

clear_output()


# @markdown Select your desired models below.

# @markdown ---
# @markdown ### **HuggingFace Token (Optional but Recommended)**
# @markdown Adding a free token prevents rate-limiting and stalling on massive downloads. Get yours at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
hf_token = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Text Encoders**
text_encoder_model = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" # @param ["qwen3vl_32b_minimax_h3-Q4_K_M.gguf", "qwen3vl_32b_minimax_h3_int4_convrot.safetensors", "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "None"]
download_custom_text_encoder = False # @param {type:"boolean"}
custom_text_encoder_url = "" # @param {type:"string"}

download_audio_vae = True
download_video_vae = True

# @markdown ---
# @markdown ### **UNet / Main Models**
unet_model = "minimax_h3_fl2va_pruned_int8_convrot.safetensors" # @param ["minimax_h3_fl2va_pruned_bf16.safetensors", "minimax_h3_fl2va_pruned_int8_convrot.safetensors", "minimax_h3_fl2va_int8_convrot.safetensors", "minimax_h3_fl2va_bf16.safetensors", "minimax_h3_ref2va_pruned_int8_convrot.safetensors", "minimax_h3_ref2va_bf16.safetensors", "MiniMax-H3-FL2VA-Q3_K_M.gguf", "MiniMax-H3-FL2VA-Q4_0.gguf", "MiniMax-H3-FL2VA-Q4_K_M.gguf", "MiniMax-H3-FL2VA-Q4_K_S.gguf", "MiniMax-H3-FL2VA-Q5_0.gguf", "MiniMax-H3-FL2VA-Q5_K_M.gguf", "MiniMax-H3-FL2VA-Q5_K_S.gguf", "MiniMax-H3-FL2VA-Q6_K.gguf", "MiniMax-H3-FL2VA-Q8_0.gguf", "MiniMax-H3-Ref2VA-Q3_K_M.gguf", "MiniMax-H3-Ref2VA-Q4_0.gguf", "MiniMax-H3-Ref2VA-Q4_K_M.gguf", "MiniMax-H3-Ref2VA-Q4_K_S.gguf", "MiniMax-H3-Ref2VA-Q5_0.gguf", "MiniMax-H3-Ref2VA-Q5_K_M.gguf", "MiniMax-H3-Ref2VA-Q5_K_S.gguf", "None"]
download_custom_unet_model = False # @param {type:"boolean"}
custom_unet_model_url = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Custom LoRAs (Optional)**
# @markdown Check the box to download the corresponding LoRA URL. You can paste any direct download link here.
download_lora_1 = True # @param {type:"boolean"}
lora_url_1 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors" # @param {type:"string"}

download_lora_2 = False # @param {type:"boolean"}
lora_url_2 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors" # @param {type:"string"}

download_lora_3 = False # @param {type:"boolean"}
lora_url_3 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors" # @param {type:"string"}

download_lora_4 = False # @param {type:"boolean"}
lora_url_4 = "" # @param {type:"string"}

download_lora_5 = False # @param {type:"boolean"}
lora_url_5 = "" # @param {type:"string"}


import os
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download

# Ensure environment is set to ComfyUI root directory
%cd /content/ComfyUI

# Enable Rust-accelerated fast transfers
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

def download_file(repo_id, repo_relative_path, token):
    if not repo_relative_path or "None" in repo_relative_path:
        return

    # Target path inside ComfyUI/models/
    destination_file = os.path.join("models", repo_relative_path)
    filename_only = os.path.basename(repo_relative_path)

    if os.path.exists(destination_file):
        print(f"⏭️ Skipping {filename_only} (Already exists in models/)\n")
        return

    print(f"Downloading {filename_only} from {repo_id} via hf_transfer...")
    try:
        hf_hub_download(
            repo_id=repo_id,
            filename=repo_relative_path,
            local_dir="models",
            token=token if token else None # Use token if provided
        )
        print(f"✅ Successfully downloaded {filename_only}\n")
    except Exception as e:
        print(f"❌ Error downloading {filename_only}: {str(e)}\n")

def download_custom_url(url, dest_folder, token=""):
    if not url or not str(url).strip():
        return

    os.makedirs(dest_folder, exist_ok=True)

    # Extract filename from URL
    filename = url.split('/')[-1].split('?')[0]
    if not filename:
        filename = "downloaded_model.safetensors"

    dest_path = os.path.join(dest_folder, filename)

    if os.path.exists(dest_path):
        print(f"⏭️ Skipping {filename} (Already exists in {dest_folder}/)\n")
        return

    print(f"Downloading {filename} from URL to {dest_folder}/...")
    headers = []

    # Securely append HF token if downloading from HuggingFace
    if token and "huggingface.co" in url:
        headers.extend(["--header", f"Authorization: Bearer {token}"])

    try:
        subprocess.run(["wget", "-q", "--show-progress", "-c", url, "-O", dest_path] + headers, check=True)
        print(f"✅ Successfully downloaded {filename}\n")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error downloading {filename}. Check the URL.\n")


# Build execution list
download_queue = []

# List of models that specifically exist on the Comfy-Org repo
comfy_org_models = [
    "qwen3vl_32b_minimax_h3_int4_convrot.safetensors",
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "minimax_h3_fl2va_pruned_bf16.safetensors",
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    "minimax_h3_fl2va_int8_convrot.safetensors",
    "minimax_h3_fl2va_bf16.safetensors",
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors",
    "minimax_h3_ref2va_bf16.safetensors"
]

token_str = hf_token.strip()

# 1. Handle Text Encoder Download
if download_custom_text_encoder and custom_text_encoder_url.strip():
    download_custom_url(custom_text_encoder_url.strip(), "models/text_encoders", token_str)
elif text_encoder_model != "None":
    repo = "Comfy-Org/MiniMax-H3" if text_encoder_model in comfy_org_models else "Abiray/MiniMax-H3-GGUF"
    download_queue.append((repo, f"text_encoders/{text_encoder_model}"))

# 2. Handle VAE Downloads
if download_audio_vae:
    download_queue.append(("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors"))

if download_video_vae:
    download_queue.append(("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors"))

# 3. Handle UNet / Main Model Download
if download_custom_unet_model and custom_unet_model_url.strip():
    clean_url = custom_unet_model_url.strip().split('?')[0]
    # Route GGUFs to models/unet/ and Safetensors to models/diffusion_models/
    target_dir = "models/unet" if clean_url.endswith(".gguf") else "models/diffusion_models"
    download_custom_url(custom_unet_model_url.strip(), target_dir, token_str)
elif unet_model != "None":
    repo = "Comfy-Org/MiniMax-H3" if unet_model in comfy_org_models else "Abiray/MiniMax-H3-GGUF"
    folder = "diffusion_models" if unet_model in comfy_org_models else "unet"
    download_queue.append((repo, f"{folder}/{unet_model}"))

# Execute Hub downloads
print("Initializing High-Speed Download Queue...\n")

for repo, rel_path in download_queue:
    download_file(repo, rel_path, token_str)

# Execute LoRA URL downloads
print("Initializing Custom LoRA Downloads...\n")
loras_to_download = [
    (download_lora_1, lora_url_1),
    (download_lora_2, lora_url_2),
    (download_lora_3, lora_url_3),
    (download_lora_4, lora_url_4),
    (download_lora_5, lora_url_5),
]

for should_download, lora_url in loras_to_download:
    if should_download and lora_url.strip():
        download_custom_url(lora_url.strip(), "models/loras", token_str)

clear_output()

# @title 3. 🚀 Run ComfyUI
use_cloudflare = True # @param {type:"boolean"}
use_interface_in_cell = False # @param {type:"boolean"}

# @markdown ### **VRAM Management**
# @markdown Select how you want ComfyUI to handle model loading:
# @markdown - **High VRAM (Keep loaded):** Keeps models in VRAM for faster subsequent generation.
# @markdown - **Normal VRAM (Load/Unload):** Offloads models from VRAM after use to save memory. (Default)
vram_management = "High VRAM (Keep loaded)" # @param ["Normal VRAM (Load/Unload)", "High VRAM (Keep loaded)"]

import torch
import os
from IPython.display import clear_output

clear_output()

%cd /content/ComfyUI

# Configure Launch Args based on selection
launch_args = "--enable-cors-header"

if vram_management == "High VRAM (Keep loaded)":
    launch_args += " --highvram"

if use_cloudflare:
    if not os.path.exists("cloudflared-linux-amd64.deb"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
        !dpkg -i cloudflared-linux-amd64.deb

    import subprocess
    import threading
    import time
    import socket

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        print("\nComfyUI finished loading, launching Cloudflare tunnel...\n")

        p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        for line in p.stderr:
            l = line.decode()
            if "trycloudflare.com " in l:
                print("This is your ComfyUI URL:", l[l.find("http"):], end='')
        clear_output()

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

elif use_interface_in_cell:
    import threading
    import time
    import socket

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        from google.colab import output
        output.serve_kernel_port_as_iframe(port, height=1024)
        clear_output()
        print("To open in a standalone window click here:")
        output.serve_kernel_port_as_window(port)

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

else:
    import socket, time, threading
    from google.colab import output

    def link_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        clear_output()
        print("Click the link below to launch the ComfyUI interface:")
        output.serve_kernel_port_as_window(port)

    threading.Thread(target=link_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 63622, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 63622 (delta 119), reused 90 (delta 90), pack-reused 63480 (from 2)
Receiving objects: 100% (63622/63622), 97.72 MiB | 16.43 MiB/s, done.
Resolving deltas: 100% (44232/44232), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 135.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 123.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 MB 54.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 88.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 61.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 MB 67.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.2/75.2 MB 60.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 MB 63.6 MB/s  0:00:01
   